In [2]:
API_KEY = ""
PINECONE_API_KEY = ""

## 8.1 보험 상품 요약서 기반 PDF 파일 기반 합성 데이터셋 생성

In [3]:
import pandas as pd
from llmops_lib.synthetic_dataset import PDFSyntheticDatasetGenerator

# pdf 파일 경로
pdf_paths = [
    "../dataset/주택화재보험_상품요약서.pdf",
    "../dataset/자동차보험_상품요약서.pdf",
    "../dataset/실손의료비보험_상품요약서.pdf",
]

# 파일을 순회하면서 합성 데이터셋 생성
df_list = []
for pdf_path in pdf_paths:
    generator = PDFSyntheticDatasetGenerator(pdf_path, API_KEY, PINECONE_API_KEY)
    df = generator.generate_dataset(sample_size=3, testset_size=5)
    df_list.append(df)

# 각 파일별 합성 데이터셋을 하나로 합침
dataset = pd.concat(df_list, ignore_index=True)
    

Applying SummaryExtractor:  83%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                          | 5/6 [00:03<00:00,  2.18it/s]Property 'summary' already exists in node '81a20a'. Skipping!
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|                                                                                                                                    | 0/6 [00:00<?, ?it/s]Property 'summary_embedding' already exists in node '727314'. Skipping!
Property 'summary_embedding' already exists in node 'b3786e'. Skipping!
Property 'summary_embedding' already exists in node '81a20a'. Skipping!
Applying SummaryExtractor:  67%|██████████████████████████████████████████████████████████████████████████████████████████████████████████                                                     | 4/6 [00:03<00:01,  1.73it/s]Property 'summary' already exists in node '7b774c'. Skippin

## 8.2 생성된 데이터셋 정보 출력

In [5]:
print(len(dataset))
print(dataset.columns)


15
Index(['user_input', 'reference_contexts', 'reference', 'synthesizer_name'], dtype='object')


## 8.3 합성 데이터셋 기반 데이터 엔트리 추가

In [6]:
import pandas as pd
from llmops_lib.dataset_storage import DatasetStorage

# 질문, 답변을 배열로 변환
question_list = dataset["user_input"].to_list()
answer_list = dataset["reference"].to_list()

# 데이터 저장소 연결
ds = DatasetStorage()
# 보험 챗봇용 데이터셋 가져오기
insurance_dataset = ds.get_dataset("insurance")

# 데이터 엔트리 추가
for question, answer in zip(question_list, answer_list):
    insurance_dataset.add_entry(
        input_variables={"question": question},
        reference_output=answer
    )